In [1]:
import requests
from bs4 import BeautifulSoup
import time
import random
import json
from urllib.parse import quote
import datetime

# --- Configurare ---
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
jobs_list = []
unique_job_links = set()
output_file = 'E-Commerce_jobs_germany6month_19_10.json'
autosave_interval = 300  # 5 minute

germany_cities = ["Germany"]
keywords = [
    "E-Commerce Marketing Specialist", "Online Sales Specialist", "E-Commerce Spezialist", "Sales Specialist E-Commerce",
    "Marketplace Manager assistant", "Amazon Marketplace Manager ", "E-Commerce Specialist", "Marketplace Manager",
    "Customer Service Specialist Logistik & E-Commerce", "Kaufmännischer Mitarbeiter Schwerpunkt Auftragsbearbeitung",
    "Customer Service Mitarbeiter in E-Commerce", "Werkstudent/in Customer Care E-Commerce", "E-Commerce Support Specialist",
    "Customer Service Specialist (Logistik & E-Commerce)", "Junior E-Commerce Manager", "Fachkraft Lagerlogistik Fulfillment / E-Commerce",
    "mitarbeiter/in e-commerce / e-commerce medewerker", "e-commerce manager / online shop manager", "online marketing manager", 
    "product data / katalog manager", "webshop/shop content manager", "Sachbearbeiter E Commerce", 
    "Digital Sales Spezialist / Online Sales Spezialist", "E-Commerce Marketing Spezialist"
]

def save_data_to_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def get_user_choice():
    # ... (Funcția rămâne neschimbată)
    print("🤖 Bun venit! Alegeți perioada pentru căutarea de joburi:")
    print("="*50)
    print("1: Ultima lună (o singură căutare)")
    print("2: Ultimele 3 luni (rulează 3 căutări: pt 1, 2, și 3 luni)")
    print("3: Ultimele 6 luni (rulează 6 căutări: pt 1, 2...6 luni)")
    print("4: Perioadă personalizată (ex: ultimele 4 luni)")
    print("5: Oricând (toate joburile, o singură căutare)")
    print("="*50)
    time_options = {
        "1_month": {"label": "Ultima lună", "seconds": "r2592000"},
        "anytime": {"label": "Oricând", "seconds": ""}
    }
    filters_to_process = []
    while True:
        choice = input("Introduceți numărul opțiunii (1-5): ")
        if choice == "1":
            print("\n✅ Ați ales: Căutare pentru ultima lună.")
            filters_to_process.append(time_options["1_month"])
            break
        elif choice == "2":
            print("\n✅ Ați ales: Căutare secvențială pentru ultimele 3 luni.")
            for i in range(1, 4):
                seconds = i * 2592000
                filters_to_process.append({"label": f"Ultimele {i} luni", "seconds": f"r{seconds}"})
            break
        elif choice == "3":
            print("\n✅ Ați ales: Căutare secvențială pentru ultimele 6 luni.")
            for i in range(1, 7):
                seconds = i * 2592000
                filters_to_process.append({"label": f"Ultimele {i} luni", "seconds": f"r{seconds}"})
            break
        elif choice == "4":
            while True:
                try:
                    num_months = int(input("Introduceți numărul de luni pentru căutare (ex: 4): "))
                    if num_months > 0:
                        print(f"\n✅ Ați ales: Căutare secvențială pentru ultimele {num_months} luni.")
                        for i in range(1, num_months + 1):
                            seconds = i * 2592000
                            filters_to_process.append({"label": f"Ultimele {i} luni", "seconds": f"r{seconds}"})
                        break
                    else:
                        print("❌ Vă rugăm introduceți un număr pozitiv.")
                except ValueError:
                    print("❌ Input invalid. Vă rugăm introduceți un număr întreg.")
            break
        elif choice == "5":
            print("\n✅ Ați ales: Căutare pentru 'Oricând'.")
            filters_to_process.append(time_options["anytime"])
            break
        else:
            print("❌ Opțiune invalidă. Vă rugăm alegeți un număr între 1 și 5.")
    return filters_to_process

# ### FUNCȚIE ACTUALIZATĂ ###
def extract_job_details(job_soup):
    """Extrage detalii specifice de pe pagina individuală a unui job, fiind atent la mai multe limbi."""
    details = {
        'description': 'N/A',
        'publishedAt': 'N/A',
        'contractType': 'N/A',
        'experienceLevel': 'N/A', # Câmp nou
        'workType': 'N/A',
        'sector': 'N/A',
        'salary': 'N/A' # Câmp nou
    }

    # 1. Descriere
    desc_block = job_soup.find('div', class_='description__text')
    if desc_block:
        details['description'] = desc_block.get_text(separator='\n', strip=True)

    # 2. Data postării
    posted_time_element = job_soup.find('span', class_='posted-time-ago__text')
    if posted_time_element:
        details['publishedAt'] = posted_time_element.text.strip()
        
    # 3. Salariu (adesea într-un element separat)
    salary_element = job_soup.find('div', class_='salary-main-rail-card__salary-info')
    if not salary_element: # Încearcă un alt selector comun
         salary_element = job_soup.find('span', class_='job-details-jobs-unified-top-card__salary-insight-text')
    if salary_element:
        details['salary'] = salary_element.get_text(strip=True).replace('\n', ' ').strip()

    # 4. Criterii (Tip contract, Experiență etc.)
    criteria_list = job_soup.find('ul', class_='description__job-criteria-list')
    if criteria_list:
        criteria_items = criteria_list.find_all('li', class_='description__job-criteria-item')
        for item in criteria_items:
            header = item.find('h3', class_='description__job-criteria-subheader')
            text = item.find('span', class_='description__job-criteria-text')
            if header and text:
                header_text = header.text.strip().lower()
                text_value = text.text.strip()
                
                # Caută Nivel de experiență (engleză sau germană)
                if 'seniority level' in header_text or 'karrierestufe' in header_text:
                    details['experienceLevel'] = text_value
                # Caută Tipul de contract (engleză sau germană)
                elif 'employment type' in header_text or 'anstellungsart' in header_text:
                    details['contractType'] = text_value
                # Caută Domeniul de activitate
                elif 'job function' in header_text or 'tätigkeitsbereich' in header_text:
                    details['workType'] = text_value
                # Caută Industriile
                elif 'industries' in header_text or 'branchen' in header_text:
                    details['sector'] = text_value
    return details


# --- Etapa Unică: Căutare, Extragere și Salvare ---
filters_to_run = get_user_choice()
time_start = time.time()
last_save_time = time_start
total_jobs_processed = 0

for filter_details in filters_to_run:
    time_param = filter_details['seconds']
    time_label = filter_details['label']
    print(f"\n--- 🔍 Încep căutarea pentru perioada: '{time_label}' ---")

    for city in germany_cities:
        for kw in keywords:
            for start in range(0, 250, 25):
                search_url = f'https://www.linkedin.com/jobs/search/?keywords={quote(kw)}&location={quote(city)}&f_TPR={time_param}&start={start}'
                print(f"--- Accesez pagina de căutare: {kw} (pagina {start//25 + 1}) ---")
                
                try:
                    response = requests.get(search_url, headers=headers, timeout=15)
                    response.raise_for_status()
                    soup = BeautifulSoup(response.text, 'html.parser')
                    job_cards = soup.find_all('div', class_='base-card')
                    
                    if not job_cards:
                        break

                    for job in job_cards:
                        link_element = job.find('a', class_='base-card__full-link')
                        clean_link = link_element['href'].split('?')[0] if link_element else ''

                        if clean_link and clean_link not in unique_job_links:
                            unique_job_links.add(clean_link)
                            total_jobs_processed += 1
                            
                            print(f"    -> Procesez jobul {total_jobs_processed}: {clean_link.split('/')[-1]}")

                            try:
                                job_response = requests.get(clean_link, headers=headers, timeout=12)
                                job_response.raise_for_status()
                                job_soup = BeautifulSoup(job_response.text, 'html.parser')
                                
                                title = job.find('h3', class_='base-search-card__title').text.strip()
                                company = job.find('h4', class_='base-search-card__subtitle').text.strip()
                                location = job.find('span', class_='job-search-card__location').text.strip()
                                
                                job_details = extract_job_details(job_soup)

                                jobs_list.append({
                                    'Title': title,
                                    'Company': company,
                                    'Location': location,
                                    'Link': clean_link,
                                    'Keyword': kw,
                                    'City_Searched': city,
                                    'Time_Filter': time_label,
                                    'Published_At': job_details['publishedAt'],
                                    'Contract_Type': job_details['contractType'],
                                    'Experience_Level': job_details['experienceLevel'], # CÂMP NOU
                                    'Work_Type': job_details['workType'],
                                    'Sector': job_details['sector'],
                                    'Salary': job_details['salary'], # CÂMP NOU
                                    'Description': job_details['description']
                                })
                                
                                ### PAUZA 1 ###
                                time.sleep(random.uniform(8, 18))

                            except requests.exceptions.RequestException as e:
                                print(f"      ❌ Eroare la accesarea paginii jobului {clean_link}: {e}")
                    
                    # ### PAUZA 2 ###
                    print(f"--- Pauză după procesarea paginii {start//25 + 1} ---")
                    time.sleep(random.uniform(5, 12))
                                
                    if time.time() - last_save_time > autosave_interval:
                        print(f"\n--- ⏱️ AUTOSAVE: Salvez {len(jobs_list)} joburi în '{output_file}'... ---")
                        save_data_to_json(jobs_list, output_file)
                        last_save_time = time.time()

                except requests.exceptions.RequestException as e:
                    print(f"❌ Eroare la accesarea paginii de căutare {search_url}: {e}")
                    time.sleep(15)

# --- Salvarea finală ---
if jobs_list:
    print(f"\n💾 Salvare finală: Salvez toate cele {len(jobs_list)} rezultate în fișierul '{output_file}'...")
    save_data_to_json(jobs_list, output_file)
    print(f"\n✅ Gata! Datele au fost salvate în fișierul '{output_file}'.")
else:
    print("\n🤷‍♂️ Nu au fost găsite joburi. Scriptul se va încheia.")

time_end = time.time()
total_duration_seconds = time_end - time_start
duration_formatted = str(datetime.timedelta(seconds=total_duration_seconds))
print(f"\n⏱️ Cautarea a durat: {duration_formatted}")

🤖 Bun venit! Alegeți perioada pentru căutarea de joburi:
1: Ultima lună (o singură căutare)
2: Ultimele 3 luni (rulează 3 căutări: pt 1, 2, și 3 luni)
3: Ultimele 6 luni (rulează 6 căutări: pt 1, 2...6 luni)
4: Perioadă personalizată (ex: ultimele 4 luni)
5: Oricând (toate joburile, o singură căutare)


Introduceți numărul opțiunii (1-5):  3



✅ Ați ales: Căutare secvențială pentru ultimele 6 luni.

--- 🔍 Încep căutarea pentru perioada: 'Ultimele 1 luni' ---
--- Accesez pagina de căutare: E-Commerce Marketing Specialist (pagina 1) ---
    -> Procesez jobul 1: e-commerce-marketing-automation-specialist-m-w-d-at-hrrk-recruiting-4304411840
    -> Procesez jobul 2: digital-marketing-specialist-m-f-d-at-allunity-4305075415
    -> Procesez jobul 3: trade-marketing-specialist-ecom-digital-shelf-m-w-d-at-brown-forman-4289472676
    -> Procesez jobul 4: search-specialist-d%C3%BCsseldorf-all-gender-at-digitl-4305075489
    -> Procesez jobul 5: search-specialist-hamburg-all-genders-at-digitl-4305077460
    -> Procesez jobul 6: search-specialist-m%C3%BCnchen-all-gender-at-digitl-4311461374
    -> Procesez jobul 7: ad-technology-consultant-d%C3%BCsseldorf-all-genders-at-digitl-4312443256
    -> Procesez jobul 8: ad-technology-consultant-hamburg-all-genders-at-digitl-4305827376
    -> Procesez jobul 9: senior-localisation-specialist-dutc